In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from IPython.display import display, Markdown

from surrogate_models import load_neutron_stars
from pathlib import Path

ARTIFACTS = Path("../../var/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

In [ ]:
FEATURES= ['rhoc', 'lambda', 'beta']
TARGETS = ['M', 'D']
COLUMNS = FEATURES + TARGETS
df: pd.DataFrame = load_neutron_stars()[COLUMNS]

## Structural sanity

- Shape and dtypes
- Missing / non finite, count per column
- DUplicates: exact duplicate rows and duplicate `(beta, lambda, rhoc)` keys with different `(M, D)` -- the latter would mean map isn't single valued

In [ ]:
dups = df.duplicated(subset=FEATURES, keep=False)
assert (df['rhoc'] > 0).all(), "All rhoc values must be positive"
assert df.isna().sum().sum() == 0, "There are missing values in the dataset"
assert dups.sum() == 0, "There are duplicate rows in the dataset"

In [ ]:
print("shape:", df.shape)
print("non-finite per col:\n",(~np.isfinite(df.select_dtypes("number"))).sum())
print("rows sharing (beta, lambda, rhoc):", dups.sum())

## Univariate stats

For each of the `5` variables (`beta`, `lambda`, `rhoc`, `M`, `D`) compute the: count, min, max, mean, median, std, 1/5/25/50/75/95/99 percentiles, **skewness**, **kurtosis**, and `n_unique`

What we are looking for:

- `rhoc ~ 1e14` and `D ~ 1e-7` span orders of magnitude certainly meaning the need of either **log transformation** or use of scaller. `M ~ O(1)` does not- Skew/kurtosis will confirm
- `n_unique` on `beta` and `lambda` tells wether inputs are **discrete grid** or continuously sampled

In [ ]:
desc = df[COLUMNS].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc['skew'] = df[COLUMNS].skew()
desc['kurt'] = df[COLUMNS].kurtosis()
desc['n_unique'] = df[COLUMNS].nunique()
print(desc)

## Discover sampling design

- `value_counts()` on `beta` and `lambda`. Is it a **Cartesian grid** (a handful of `beta` vs `lamdba` each with a long `rhoc` sweep) 
- Cross-tabulate: for each `(beta, lambda)` pai how many `rhoc` points and over what range

### Why this decides validation strategy

- If it's a dense grid and you do a random 80/20 split, every test point sits between two near-identical training points. You'll get R² ≈ 0.9999 that proves nothing — it only shows you can interpolate between adjacent grid rows. A reviewer will reject that.
- The honest test of "does this replace the PDE solver" is leave-whole-(beta,lambda)-groups-out (interpolation in parameter space) and hold out the edges of the `rhoc`/`beta`/`lambda` ranges (extrapolation). Plan your splits now, from the grid structure.

| Level                 | Question (yes/no)                                            | Test / Artifact                                   | Split implication                                     |
| --------------------- | ------------------------------------------------------------ | ------------------------------------------------- | ----------------------------------------------------- |
| **L1**                | Is it a (near-)Cartesian grid?                               | Fill ratio + count-pivot heatmap                  | Random split forbidden if yes                         |
| **L2a** *(Deductive)* | Does `n_β · n_λ · n_ρ = N`?                                  | `52.6%` → **No, partial**                         | Rules out naive random split anyway                   |
| **L2b** *(Inductive)* | Is `count(rhoc) per (β,λ)` \~ constant?                      | `df.groupby([β,λ]).size().describe()` + histogram | Constant ⇒ rectangular; variable ⇒ ragged             |
| **L2c** *(Deductive)* | Do all pairs share the **same** `rhoc` set?                  | Signature hash per pair                           | Same set ⇒ true grid; different sets ⇒ per-pair sweep |
| **L2d** *(Abductive)* | Are missing pairs at the **edges** or **interior**?          | Mask heatmap of the pivot                         | Tells you if extrapolation regions are already sparse |
| **L3.1**              | Which `(β,λ)` pairs are **safe holdouts** for interpolation? | Interior pairs with typical `rhoc` coverage       | → GroupKFold on `(β,λ)`                               |
| **L3.2**              | Which rows are **edge extrapolation** candidates?            | Top/bottom quantiles of `β`, `λ`, `rhoc`          | → Explicit edge-holdout set                           |


In [ ]:
X = ["beta", "lambda", "rhoc"]

# --- L2a: Cartesian fill ratio -------------------------------------------
n = {c: df[c].nunique() for c in X}
full = np.prod(list(n.values()))
fill = len(df) / full
display(Markdown(
    f"**Cartesian check:** {len(df):,} rows / {full:,} full grid "
    f"= **{fill:.1%} fill** — cardinalities {n}"
))

# --- L2b: points per (beta, lambda) --------------------------------------
pair = df.groupby(["beta", "lambda"]).size().rename("n_rhoc")
display(Markdown("**Points per (β,λ) pair — distribution:**"))
display(pair.describe().to_frame().T.style.format("{:.1f}"))

In [ ]:
# --- L2b: points per (beta, lambda) --------------------------------------
pair = df.groupby(["beta", "lambda"]).size().rename("n_rhoc")
display(Markdown("**Points per (β,λ) pair — distribution:**"))
display(pair.describe().to_frame().T.style.format("{:.1f}"))

In [ ]:
pivot = df.pivot_table(index="beta", columns="lambda",
                       values="rhoc", aggfunc="count")

fig, ax = plt.subplots(figsize=(6.5, 3.6), dpi=150)  # ~half of 13.33x7.5 slide

sns.heatmap(
    pivot,
    mask=pivot.isna(),
    cmap="viridis",
    cbar_kws={"label": "n(rhoc)", "shrink": 0.75, "pad": 0.02},
    linewidths=0,
    ax=ax,
)

ax.set_xlabel("λ"); ax.set_ylabel("β")
ax.set_title("Sampling density per (β, λ) — white = unsampled", fontsize=10)

# Thin out tick labels so 50×37 grid stays readable
ax.set_xticks(ax.get_xticks()[::4]); ax.set_yticks(ax.get_yticks()[::5])
ax.tick_params(axis="both", labelsize=8, length=2)
plt.setp(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
# ppt_save(fig, "sampling_grid_compact")   # SVG + PNG via your shared helper
plt.show()

### The missing corner

The corner is excluded from the physical parameter domain because TODO Review:

- Superluminal sound speed -- $c_s^2 > c^2$ at high central density; EoS becomes acausal, code refuses to integrate
- Thermodynamic instability -- $dP/d\rho \leq 0$ locally, star has no equilibirum
- TOV non-convergence: solver hits the maximum mass turnover before reaching the sampled `rhoc` or shooting method diverges

As consiquence: the surrogate model is defined on the sampled convex hull.

#### Declared domain

The surrogate is valid on the convex hull of sampled (`beta`, `lambda`) pairs, exlcuding the high-$\beta \times \lambda$ corner $(\beta \geq 15, \lambda \geq 2.8)$ where... TODO Reason

Predictions outside the hull are undefined and the API refuses them via `in_domain()`

In [ ]:
from scipy.spatial import ConvexHull, Delaunay

# Convex hull of *sampled* (beta, lambda) pairs
pairs = df[["beta", "lambda"]].drop_duplicates().to_numpy()
hull  = Delaunay(pairs)  # for point-in-hull tests

def in_domain(beta, lam):
    return hull.find_simplex(np.c_[beta, lam]) >= 0

df["in_hull"] = in_domain(df["beta"].values, df["lambda"].values)  # all True by construction

In [ ]:
# Signature = hashed tuple of sorted unique rhoc values per pair
sig = (df.groupby(["beta", "lambda"])["rhoc"]
         .agg(lambda s: hash(tuple(np.sort(s.unique())))))
print(f"Distinct rhoc-sweep signatures across pairs: {sig.nunique()}")

rhoc_span = (df.groupby(["beta", "lambda"])["rhoc"]
               .agg(n="nunique", lo="min", hi="max"))
display(rhoc_span.describe().style.format("{:.3g}"))

| Column                                      | Reading                              | Physical meaning                                                  |
| ------------------------------------------- | ------------------------------------ | ----------------------------------------------------------------- |
| `n` mean 139, min **6**, max 231, std 80    | Highly variable point count per pair | Some pairs truncate almost immediately (6 points → unusable)      |
| `lo` std = **0**, all = 5.89e14             | Universal lower anchor               | Sweep starts at a fixed floor (below neutron-drip? config choice) |
| `hi` std = 9.83e14, min 6.7e14, max 3.34e15 | Upper limit varies 5×                | Solver stops at max-mass turnover; softer EoS turns over sooner   |
| 25%+ pairs reach `hi = 3.34e15`             | Full sweep only for a subset         | Stiff-EoS region survives to high density                         |
| 1 610 pairs sampled                         | ≤ 50×37 = 1 850 possible             | Confirms the missing corner from the pivot                        |

### Bucket classification of pairs 

Rough from quartiles

| Bucket          | `n_rhoc` | Fraction | Use                                 |
| --------------- | -------- | -------- | ----------------------------------- |
| Truncated-early | 6–58     | \~25%    | Risk for training; consider filter  |
| Mid-sweep       | 59–230   | \~50%    | Interior training pool              |
| Full-sweep      | 231      | \~25%    | Anchor for extrapolation-in-ρ tests |


In [ ]:
rhoc_span = (df.groupby(["beta","lambda"])["rhoc"]
               .agg(n="nunique", lo="min", hi="max")).reset_index()


fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), dpi=150)

# (a) how ragged is the sweep length?
axes[0].hist(rhoc_span["n"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(20, color="crimson", ls="--", lw=1, label="min-usable (n=20)")
axes[0].set(xlabel="unique rhoc per (β,λ)", ylabel="pairs",
            title="Sweep length distribution")
axes[0].legend(fontsize=8)

# (b) where does the sweep terminate? → the physics turnover map
hi_grid = rhoc_span.pivot(index="beta", columns="lambda", values="hi")
sns.heatmap(hi_grid / 1e15, mask=hi_grid.isna(), cmap="magma",
            cbar_kws={"label": "rhoc_max [1e15]", "shrink": 0.75},
            ax=axes[1])
axes[1].set_title("Truncation point of rhoc sweep per (β, λ)")
axes[1].tick_params(labelsize=8)

plt.tight_layout(); plt.show()

### Data sweep

Sweep is left-anchored (`rhoc_min` shared , `std=0`) and right truncated per pair (`rhoc_max` varies 5 times, correlates with EoS stiffness). This matches the missing corner both are signatures of TOV turnover.

Splitter update, exclude `n_rhoc < 20` paris 

In [ ]:
MIN_N_RHOC = 20                                # tune from panel (a)
good_pairs = rhoc_span.loc[rhoc_span["n"] >= MIN_N_RHOC, ["beta","lambda"]]
df_use = df.merge(good_pairs, on=["beta","lambda"], how="inner")
print(f"kept {len(df_use):,}/{len(df):,} rows across "
      f"{len(good_pairs):,}/{len(rhoc_span):,} pairs")

## Relationships and visualization

### Mass vs rhoc colored by (beta, lambda)

he headline plot. Expect each curve to rise to a maximum mass and turn over (stable vs. unstable branch). Confirm whether  changes sign in your data. That turnover is a high-curvature region where simple models fail and where you'll want to report error separately. Decide whether to keep, segment, or flag the unstable branch.

### D vs rhoc and D vs M

Establish monotonicity and scale

### Correlation matrix

Compute both `Pearson` and `Spearman` (linear and rank/monotonic). Relationships are nonolinear, so Spearman is the most honest summary

### Pariplot / hexbin of the 5 vairables on log axes where appropiate

### Sensitivity

How much do `M` and `D` move with `beta` and `lambda` at fixed `rhoc`? If they barely respond the regression is "easy", if response is sharp -- thats where error will concentrate.

| #      | Question (yes/no)                              | Artifact                                                                         | Decision it drives             |
| ------ | ---------------------------------------------- | -------------------------------------------------------------------------------- | ------------------------------ |
| **P1** | Does every pair have a turnover?               | `dM/dρ` sign-change table + M–ρ family plot coloured by λ, faceted by β-quantile | Segment stable/unstable branch |
| **P2** | Is `D(ρ)` monotone increasing per pair?        | Per-pair Spearman(ρ, D) histogram                                                | Log-transform `D` for training |
| **P3** | Is `D` monotone in `M` on the *stable* branch? | Per-pair Spearman(M, D)ₛₜₐᵦₗₑ                                                    | Use `M` as a feature for `D`   |
| **P4** | Which coordinate dominates variance?           | Spearman heatmap + partial-Spearman                                              | Feature importance sanity      |
| **P5** | Where is response sharpest?                    | Local sensitivity ∂M/∂β, ∂M/∂λ at fixed `rhoc` bins                              | Where error will concentrate   |

In [ ]:
sns.set_context("notebook", font_scale=0.9)

PAIR = ["beta", "lambda"]
df_use = df_use.sort_values(PAIR + ["rhoc"]).reset_index(drop=True)
pair_key = df_use[PAIR].astype(str).agg("|".join, axis=1)
df_use["pair_id"] = pair_key

In [ ]:
def pair_diagnostics(g):
    g = g.sort_values("rhoc")
    dM = np.diff(g["M"].values)
    sign_changes = int(np.sum(np.diff(np.sign(dM)) != 0))
    idx_max = g["M"].idxmax()
    return pd.Series({
        "n":             len(g),
        "M_max":         g["M"].max(),
        "rhoc_at_Mmax":  g.loc[idx_max, "rhoc"],
        "sign_changes":  sign_changes,
        "has_turnover":  sign_changes >= 1,
        "monotone_D":    (np.diff(g["D"].values) >= 0).all(),
    })

diag = (df_use.groupby(PAIR, group_keys=False)
              .apply(pair_diagnostics).reset_index())
diag.to_parquet(ARTIFACTS / "pair_diagnostics.parquet")

display(Markdown("**P1 verdict — turnover per pair:**"))
display(diag[["has_turnover","monotone_D"]].mean()
              .to_frame("fraction_of_pairs").style.format("{:.1%}"))
display(diag["sign_changes"].value_counts().sort_index()
              .to_frame("n_pairs"))

In [ ]:
# Bin pairs by lambda quartile for colour; facet by beta quartile → 4 panels max
diag["beta_q"]   = pd.qcut(diag["beta"],   4, labels=["β-Q1","β-Q2","β-Q3","β-Q4"])
diag["lambda_q"] = pd.qcut(diag["lambda"], 4, labels=["λ-Q1","λ-Q2","λ-Q3","λ-Q4"])
df_p = df_use.merge(diag[PAIR + ["beta_q","lambda_q"]], on=PAIR)

g = sns.relplot(
    data=df_p, x="rhoc", y="M", hue="lambda_q",
    col="beta_q", col_wrap=2, kind="line",
    units="pair_id", estimator=None,
    lw=0.4, alpha=0.35, height=2.8, aspect=1.4,
    palette="viridis", facet_kws={"sharex": True, "sharey": True},
)
g.set(xscale="log")
# Overlay turnover points
for ax, (bq, sub) in zip(g.axes.flat, diag.groupby("beta_q")):
    ax.scatter(sub["rhoc_at_Mmax"], sub["M_max"], s=6, c="red", zorder=5,
               label="M_max")
g.fig.suptitle("M vs ρc — turnover marked in red", y=1.02)
g.tight_layout()

In [ ]:
from scipy.stats import spearmanr

def pair_spearman(g, x, y):
    if g[x].nunique() < 3: return np.nan
    return spearmanr(g[x], g[y]).statistic

sp_rhoc_D = (df_use.groupby(PAIR)
                   .apply(pair_spearman, "rhoc", "D")
                   .rename("spearman_rhoc_D").reset_index())

valid = sp_rhoc_D["spearman_rhoc_D"].dropna()
lo, hi = valid.min(), valid.max()
margin = max((hi - lo) * 0.1, 1e-3)
n_bins = min(40, max(1, valid.nunique()))

fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=150)
ax.hist(valid, bins=n_bins, range=(lo - margin, hi + margin),
        color="steelblue", edgecolor="white")
ax.axvline(1.0, color="green", ls="--", lw=1, label="perfect monotone")
ax.axvline(valid.median(), color="crimson",
           ls="-", lw=1, label=f"median={valid.median():.3f}")
ax.set(xlabel="Spearman(ρc, D) per pair", ylabel="pairs",
       title="P2 — D vs ρc monotonicity")
ax.legend(fontsize=8); plt.tight_layout()


In [ ]:
def stable_branch(g):
    """Rows up to (and including) M_max — the stable branch."""
    g = g.sort_values("rhoc")
    return g.iloc[:g["M"].values.argmax() + 1]

df_stable = df_use.loc[
    df_use.groupby(PAIR, group_keys=False)
          .apply(stable_branch)
          .index
]

sp_M_D = (df_stable.groupby(PAIR)
                   .apply(pair_spearman, "M", "D")
                   .rename("spearman_M_D_stable").reset_index())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), dpi=150)
valid_sp = sp_M_D["spearman_M_D_stable"].dropna()
lo, hi = valid_sp.min(), valid_sp.max()
margin = max((hi - lo) * 0.1, 1e-3)
n_bins = min(40, max(1, valid_sp.nunique()))
ax[0].hist(valid_sp, bins=n_bins, range=(lo - margin, hi + margin),
           color="darkorange", edgecolor="white")
ax[0].set(xlabel="Spearman(M, D) on stable branch", ylabel="pairs",
          title="P3 — D↔M monotonicity (stable branch)")

# Global hexbin: D vs M, log-y, stable rows only
hb = ax[1].hexbin(df_stable["M"], df_stable["D"], gridsize=60,
                  yscale="log", cmap="magma", mincnt=1)
fig.colorbar(hb, ax=ax[1], label="count")
ax[1].set(xlabel="M", ylabel="log D",
          title="D vs M — density on stable branch")
plt.tight_layout()


In [ ]:
# Phase 3 — relationships
print("\nPearson:\n",  df[COLUMNS].corr("pearson").round(3))
print("\nSpearman:\n", df[COLUMNS].corr("spearman").round(3))

fig, ax = plt.subplots(1, 2, figsize=(13,5))
for (b,l), g in df.groupby(["beta","lambda"]):
    g = g.sort_values("rhoc")
    ax[0].plot(g["rhoc"], g["M"], lw=.8, label=f"β={b},λ={l}")
    ax[1].plot(g["rhoc"], g["D"], lw=.8)
for a in ax: a.set_xscale("log")
ax[1].set_yscale("log")
ax[0].set(xlabel="rhoc", ylabel="M", title="M vs rhoc (look for the turnover)")
ax[1].set(xlabel="rhoc", ylabel="D", title="D vs rhoc")
plt.tight_layout(); plt.savefig("eda_curves.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes[0], FEATURES+ TARGETS):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
for ax, col in zip(axes[1], FEATURES + TARGETS):
    sns.histplot(np.log10(df[col].clip(lower=1e-30)), kde=True, ax=ax)
    ax.set_title(f"log10({col})")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey='row')
for i, y in enumerate(TARGETS):
    for j, x in enumerate(FEATURES):
        hue = [c for c in FEATURES if c != x][0]
        sns.scatterplot(
            data=df,
            x=x,
            y=y,
            hue=hue,
            palette="viridis",
            s=8,
            ax=axes[i, j],
            legend=(j == 2),
        )
        if x == "rhoc":
            axes[i, j].set_xscale("log")
plt.tight_layout()

In [ ]:
sns.pairplot(
    df.sample(min(2000, len(df))),
    vars=FEATURES + TARGETS,
    plot_kws={"s": 6, "alpha": 0.4},
)
print(df.corr(method="spearman"))  # Spearman > Pearson: catches monotonic non-linear


In [ ]:
X, y = df[FEATURES].copy(), df[TARGETS]
X["log_rhoc"] = np.log10(X["rhoc"])  # cheap feature-engineering probe

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "linear": make_pipeline(StandardScaler(), LinearRegression()),
    "poly3": make_pipeline(
        PolynomialFeatures(3, include_bias=False), StandardScaler(), LinearRegression()
    ),
    "rf": RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
}
for name, m in models.items():
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    for k, t in enumerate(TARGETS):
        print(
            f"{name:8s} {t}: R²={r2_score(yte[t], pred[:, k]):.4f}  "
            f"MAE={mean_absolute_error(yte[t], pred[:, k]):.4g}"
        )
